> **Note (Oct 2026):** the reported results were NOT produced with this notebook. They were run on a Mac (simulation + llama.cpp real engine) — see `docs/setup.md` and `README.md`. This notebook is kept as an optional vLLM-on-Colab path. It expects the repo uploaded as `project-repo-starter.zip` containing a `project-repo/` folder; if you zip `code_repo/` instead, change the `%cd project-repo` cell to `%cd code_repo`. For vLLM, start the gateway with `--backend vllm --capacity-tokens <from vLLM log>`.

# GPU Memory-Aware Scheduling -- Colab Runner

**Before running anything:** go to `Runtime > Change runtime type > T4 GPU` (free tier) if you want to run the REAL vLLM section (Part 2). Part 1 (simulation) works fine on CPU-only runtime too.

This notebook has two independent parts:
- **Part 1: Simulation** -- runs entirely on CPU, reproduces everything we've validated so far (no GPU needed)
- **Part 2: Real vLLM test** -- uses Colab's free T4 GPU to actually run vLLM + our gateway, for the first REAL (non-simulated) validation of the project

## Setup: upload the project repo

In [ ]:
# Upload project-repo-starter.zip using the file browser on the left,
# OR run this cell to get an upload widget:
from google.colab import files
uploaded = files.upload()  # select project-repo-starter.zip when prompted

In [ ]:
!unzip -o project-repo-starter.zip
%cd project-repo
!ls

## Part 1: Simulation (no GPU needed -- works on any runtime)

In [ ]:
!pip install -q numpy matplotlib scikit-learn pandas

In [ ]:
# Baseline vs. our memory-aware policy demo
%cd benchmarks
!python3 run_simulation.py
%cd ..

In [ ]:
# View the generated plots inline
from IPython.display import Image, display
display(Image('results/memory_usage_comparison.png'))
display(Image('results/outcome_comparison.png'))

In [ ]:
# Full evaluation suite: threshold sweep + multi-load comparison
%cd benchmarks
!python3 run_evaluation.py
%cd ..

In [ ]:
# Predictive batching experiment (train the real predictor first)
%cd scheduler
!python3 train_predictor.py
%cd ../benchmarks
!python3 run_predictive_experiment.py
%cd ..

In [ ]:
# Scheduling order optimization experiment
%cd benchmarks
!python3 run_optimization_experiment.py
%cd ..

## Part 2: Real vLLM Test (requires GPU runtime -- T4 free tier works)

**Check you have a GPU first:**

In [ ]:
!nvidia-smi
# If this errors, go to Runtime > Change runtime type > select a GPU, then re-run from the top

In [ ]:
# Install vLLM -- this takes several minutes on first run
!pip install -q vllm fastapi uvicorn httpx nest_asyncio

In [ ]:
# Start vLLM as a background process. Use a SMALL model to fit comfortably
# in the free T4's 16GB VRAM -- Qwen2.5-1.5B-Instruct is a good fit.
import subprocess, time
vllm_proc = subprocess.Popen([
    'vllm', 'serve', 'Qwen/Qwen2.5-1.5B-Instruct',
    '--max-model-len', '4096', '--port', '8000',
    '--gpu-memory-utilization', '0.85'
], stdout=open('/content/vllm.log', 'w'), stderr=subprocess.STDOUT)
print(f'vLLM starting (PID {vllm_proc.pid}), this takes 1-3 minutes to load...')
print('Run the next cell to check progress.')

In [ ]:
# Check vLLM startup progress -- re-run this cell until you see
# 'Uvicorn running on http://0.0.0.0:8000' in the output
!tail -20 /content/vllm.log

In [ ]:
# CRITICAL STEP flagged throughout our docs: verify the real KV-cache
# metric name before trusting the gateway's decisions
!curl -s http://localhost:8000/metrics | grep -i cache

In [ ]:
# Quick sanity check: send one direct request to vLLM
!curl -s http://localhost:8000/v1/completions \
  -H "Content-Type: application/json" \
  -d '{"model": "Qwen/Qwen2.5-1.5B-Instruct", "prompt": "Explain photosynthesis in one sentence.", "max_tokens": 50}'

In [ ]:
# IMPORTANT: before starting the gateway, get vLLM's REAL KV-cache capacity
# in bytes and update GPU_KV_CACHE_CAPACITY_BYTES in scheduler/vllm_integration.py.
# vLLM prints this at startup -- search the log for cache size info:
!grep -i 'GPU KV cache\|kv cache size\|num_gpu_blocks' /content/vllm.log

In [ ]:
# EDIT the two placeholder values in vllm_integration.py to match what you
# found above, then start the gateway as a background process
import subprocess
gateway_proc = subprocess.Popen(
    ['python3', 'vllm_integration.py', '--backend', 'vllm', '--policy', 'admission_control_reserved',
     '--capacity-tokens', 'REPLACE_WITH_KV_CACHE_TOKENS_FROM_VLLM_LOG'],
    cwd='scheduler', stdout=open('/content/gateway.log', 'w'), stderr=subprocess.STDOUT
)
import time; time.sleep(3)
print(f'Gateway starting (PID {gateway_proc.pid})')
!cat /content/gateway.log

In [ ]:
# Send a real request THROUGH the gateway (not directly to vLLM)
!curl -s -X POST http://localhost:8080/v1/completions \
  -H "Content-Type: application/json" \
  -d '{"model": "Qwen/Qwen2.5-1.5B-Instruct", "prompt": "What is the capital of France?", "max_tokens": 50}' \
  -w "\nHTTP status: %{http_code}\n"

In [ ]:
# Check gateway decision stats
!curl -s http://localhost:8080/gateway/stats

In [ ]:
# THE REAL EVALUATION: replay a trace through the gateway (our policy)
# and directly against vLLM (baseline, no admission control), then compare.
%cd traces
import sys; sys.path.insert(0, '.')
from generator import generate_trace, save_trace_csv
save_trace_csv(generate_trace(num_requests=100, arrival_rate_per_step=3.0, seed=7), 'colab_trace.csv')
%cd ..
print('Trace generated')

In [ ]:
# Baseline: direct to vLLM, no gateway
!python3 benchmarks/replay_trace_live.py --trace traces/colab_trace.csv \
  --target http://localhost:8000 --model Qwen/Qwen2.5-1.5B-Instruct \
  --speed 5 --output /content/baseline_results.json

In [ ]:
# Our policy: through the memory-aware gateway
!python3 benchmarks/replay_trace_live.py --trace traces/colab_trace.csv \
  --target http://localhost:8080 --model Qwen/Qwen2.5-1.5B-Instruct \
  --speed 5 --output /content/policy_results.json

In [ ]:
# Compare real results -- THIS is the first real (non-simulated) validation
import json
baseline = json.load(open('/content/baseline_results.json'))['summary']
policy = json.load(open('/content/policy_results.json'))['summary']
print(f"{'Metric':<25}{'Baseline (direct)':<20}{'Our Policy (gateway)':<20}")
for k in baseline:
    print(f"{k:<25}{str(baseline[k]):<20}{str(policy.get(k, '-')):<20}")

In [ ]:
# Cleanup: stop background processes when done
vllm_proc.terminate()
gateway_proc.terminate()
print('Stopped vLLM and gateway')